# Simple GNN Test

Test the GNN with one example

In [ ]:
%load_ext autoreload
%autoreload 1
%aimport image_processing_tools

## Image pre-processing

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer
from pathlib import Path
import matplotlib
# matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

search_path = ("~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_CY5, DAPI",
               "~/A8/Data_Roan/260228_She3_Mutants_DAPI",)
file_pattern = ("CROP_*",
                "MAX_*",
                "SHARPEST*",
                "MAX_CET145*")

config = {
    "preprocessing": {
        "resize_image": False,
        "max_dim": 1080,
        "outlier_percentile": 0.35,
        "quantization": "8bit"
    }
}

# Find the files. The 'files' variable will hold the list of Path objects.
files = find_files_by_pattern(search_path[0], file_pattern[0], verbose=True)

In [ ]:
from image_processing_tools.image_class.image_filters import ImageJProcessor
import matplotlib.pyplot as plt

crop_img = ImageContainer(files[0],config).merge()
pipeline_seg = ImageJProcessor(crop_img)
pipeline_seg.reset()
pipeline_seg.fft_bandpass_filter()
pipeline_seg.enhance_contrast_clahe(slope=1.5)
pipeline_seg.imagej_rolling_ball(radius=60)
pipeline_seg.gaussian_blur(sigma=3)
seg_img = pipeline_seg.return_image()

pipeline_intens = ImageJProcessor(crop_img)
pipeline_intens.reset()
pipeline_intens.fft_bandpass_filter()
pipeline_intens.enhance_contrast_clahe(slope=1)
pipeline_intens.gaussian_blur(sigma=5)
intens_img = pipeline_intens.return_image()

fig, axes = plt.subplots(1,2,figsize=(8,4))
axes[0].imshow(seg_img, cmap='gray'); axes[0].set_title('Image for Segmentation')
axes[1].imshow(intens_img, cmap='gray'); axes[1].set_title('Image for Intensity')

for ax in axes: ax.axis('off')
# plt.axis('off')
plt.tight_layout()
plt.show()

## Extract features from simple graph

In [ ]:
from image_processing_tools.dapi_tracing.connectivity_score import detect_nuclei, filter_nuclei, extract_graph

labels, binary_mask_filled = detect_nuclei(seg_image=seg_img, use_watershed=False)
valid_nuclei_data, valid_labels, avg_nucleus_length = filter_nuclei(labels)
nuclei_df, paths_df, edge_index = extract_graph(valid_nuclei_data, intens_img, binary_mask_filled)

In [ ]:
paths_df

In [ ]:
import pandas as pd

pd.DataFrame(edge_index)

In [ ]:
edge_label = [1,0,0,0,1,0,0,1,0,1]

In [ ]:
nuclei_df

## Create pyg object

In [ ]:
import torch

# 1. Remove the ID columns so only the pure features remain
node_features = nuclei_df.drop(columns=['node_id'])
edge_features = paths_df.drop(columns=['source_node', 'target_node'])

# 2. Convert the feature DataFrames into PyTorch tensors (typically float32)
x = torch.tensor(node_features.values, dtype=torch.float32)
edge_attr = torch.tensor(edge_features.values, dtype=torch.float32)

# 3. Convert the edge index list into a PyTorch tensor (typically long/int64)
edge_index_tensor = torch.tensor(edge_index, dtype=torch.long)

# 4. Convert the edge labels to a PyTorch tensor
edge_label_tensor = torch.tensor(edge_label, dtype=torch.float32)

# Display the resulting tensor shapes to verify
print("Node features (x) shape:", x.shape)
print("Edge features (edge_attr) shape:", edge_attr.shape)
print("Edge index shape:", edge_index_tensor.shape)
print("Edge label shape:", edge_label_tensor.shape)


In [ ]:
from torch_geometric.data import Data
import torch_geometric.transforms as T

data = Data(
    x = x,
    edge_attr = edge_attr,
    edge_index = edge_index_tensor,
    edge_label = edge_label_tensor,
)
data = T.ToUndirected()(data)

In [ ]:
# Calculate mean and standard deviation for node features across the node dimension
x_mean = data.x.mean(dim=0)
x_std = data.x.std(dim=0)
# Add a small epsilon (1e-7) to prevent division by zero for constant features
data.x = (data.x - x_mean) / (x_std + 1e-7)

# Calculate mean and standard deviation for edge features across the edge dimension
edge_mean = data.edge_attr.mean(dim=0)
edge_std = data.edge_attr.std(dim=0)
data.edge_attr = (data.edge_attr - edge_mean) / (edge_std + 1e-7)

## Train the GNN

In [ ]:
from image_processing_tools.dapi_tracing.simple_gnn import Model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = Model(hidden_channels = 32, dropout_p=0).to(device)

# Initialize the lazy layers by doing a dry-run with your first batch of data
with torch.no_grad():
    model(data.x.to(device), data.edge_index.to(device), data.edge_attr.to(device))

# Now that the weights are created, give them to the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

def train_model(model, data, optimizer, criterion):
    """Performs one epoch of training for a link classification task."""
    model.train()

    optimizer.zero_grad()

    # 1. Get predictions for all edges in the batch. The model's forward pass handles this.
    pred = model(data.x, data.edge_index, data.edge_attr)

    # 2. Ground truth labels are stored in data.edge_label
    ground_truth = data.edge_label.float()

    # 3. Calculate loss and perform backpropagation
    loss = criterion(pred, ground_truth)
    loss.backward()
    optimizer.step()

    # 4. Calculate accuracy
    pred_labels = (pred > 0.5).float()
    accuracy = (pred_labels == ground_truth).sum().item() / ground_truth.size(0)

    return loss.item(), accuracy, pred_labels

In [ ]:
import torch.nn as nn

# 1. Use BCELoss because your model already outputs probabilities (via Sigmoid)
criterion = nn.BCELoss()

# 2. Ensure your ground truth labels are floating-point numbers, 
# which is strictly required by BCELoss
if hasattr(data, 'edge_label'):
    data.edge_label = data.edge_label.float()

# 3. Training loop for 100 epochs
epochs = 50
loss_tracker = []
acc_tracker = []
pred_labels = []

for epoch in range(1, epochs + 1):
    train_data = data.to(device)
    
    loss, acc, pred_labels = train_model(model, data, optimizer, criterion)
    loss_tracker.append(loss)
    acc_tracker.append(acc)
    pred_labels = pred_labels
    # Print the loss every 10 epochs to monitor progress
    if epoch % 1 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | Accuracy: {acc:.4f}")


In [ ]:
import matplotlib.pyplot as plt

def plot_training_metrics(losses, accuracies):
    """
    Plots training loss and accuracy over epochs.
    
    Args:
        losses (list): A list of loss values per epoch.
        accuracies (list): A list of accuracy values per epoch.
    """
    # Generate an x-axis for the epochs (1, 2, 3, ...)
    epochs = range(1, len(losses) + 1)
    
    # Create a figure with 1 row and 2 columns for subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # --- Plot 1: Loss ---
    ax1.plot(epochs, losses, color='tab:red', marker='o', markersize=4, linestyle='-', linewidth=2, label='Loss')
    ax1.set_title('Model Loss')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.grid(True, linestyle='--', alpha=0.7)
    ax1.legend()
    
    # --- Plot 2: Accuracy ---
    ax2.plot(epochs, accuracies, color='tab:blue', marker='o', markersize=4, linestyle='-', linewidth=2, label='Accuracy')
    ax2.set_title('Model Accuracy')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy')
    ax2.grid(True, linestyle='--', alpha=0.7)
    ax2.legend()
    
    # Adjust layout to prevent overlapping text and display the plot
    plt.tight_layout()
    plt.show()

plot_training_metrics(loss_tracker, acc_tracker)


## Test the model

In [ ]:
import torch

def test_model(model, data, criterion):
    """Evaluates the model on a test or validation dataset."""
    # Set the model to evaluation mode (important for BatchNorm and Dropout)
    model.eval()

    # Disable gradient calculation for inference
    with torch.no_grad():
        # 1. Get predictions for all edges
        pred = model(data.x, data.edge_index, data.edge_attr)

        # 2. Ground truth labels
        ground_truth = data.edge_label.float()

        # 3. Calculate loss
        loss = criterion(pred, ground_truth)

        # 4. Calculate accuracy
        # The model outputs probabilities (via sigmoid), so we threshold at 0.5
        pred_labels = (pred > 0.5).float()
        accuracy = (pred_labels == ground_truth).sum().item() / ground_truth.size(0)

    # Return as standard Python numbers
    return loss.item(), accuracy

test_loss, test_acc = test_model(model, data, criterion)

print(test_loss,test_acc)

Conclusion: the model is tested on a simple graph and shows it can learn the edge connections effectively.